# Intraday Reinforcement Learning Trading Pipeline
## Modular Continuous Compounding PPO Agent with FinBERT Sentiment and Real-World MIS Frictions

This notebook demonstrates the complete **Sequential Data Pipeline**:
1. **File 1 (`MarketDataProcessor`)**: Ingests 5-minute OHLCV data, engineers momentum (RSI, MACD), volatility (ATR, BB), and trend (ADX) indicators, cleans, and normalizes features.
2. **File 2 (`SentimentProcessor`)**: Scores financial news headlines via `ProsusAI/finbert` to compute continuous directional sentiment in $[-1, 1]$.
3. **`final_state_vector` Merging**: Merges market features and sentiment scores on date/timestamp.
4. **Chronological 80/20 Split**: 80% training set and 20% testing set.
5. **File 3 (`ConstrainedTradingEnv`)**: Vectorized Gymnasium environment enforcing continuous compounding, 15:15 MIS auto-square-off, day-lock stop-loss penalty, 3-trade daily cap, and round-trip transaction fees.
6. **File 4 Training & Continuous Backtesting**: Trains a PPO agent and backtests on the test set in an unbroken continuous loop, auditing Sortino ratio, fees, and Net vs. Gross PnL.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in Python path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

# Import core pipeline components from src
from src.data_processing import (
    MarketDataProcessor,
    SentimentProcessor,
    create_final_state_vector,
    generate_intraday_5min_market_data,
    load_archive_headlines,
)
from src.model import ConstrainedTradingEnv, create_vectorized_env
from src.train import calculate_sortino_ratio

print("All modules loaded successfully.")

### Step 1: Market Data Processing (File 1)
We fetch and engineer technical indicators for 5-minute intraday candles.

In [ ]:
# Generate or fetch 5-min intraday market data (1,500 candles = 20 trading days)
raw_market_df = generate_intraday_5min_market_data(n_rows=1500, start_date="2024-01-02", initial_price=150.0)

market_proc = MarketDataProcessor()
market_state_df = market_proc.process_dataframe(raw_market_df)

print(f"Market State Vector Shape: {market_state_df.shape}")
market_state_df.head(3)

### Step 2: News Sentiment Scoring via FinBERT (File 2)
We load news headlines from `archive/` and score them using FinBERT or simulated distributions.

In [ ]:
# Load archive headlines or generate daily sentiment
unique_dates = sorted(list(set(pd.to_datetime(market_state_df.index).strftime("%Y-%m-%d"))))
rng = np.random.default_rng(42)

sentiment_df = pd.DataFrame({
    "date": unique_dates,
    "daily_sentiment": np.clip(rng.normal(0.05, 0.35, size=len(unique_dates)), -1.0, 1.0),
})

print(f"Daily Sentiment Rows: {len(sentiment_df)}")
sentiment_df.head(3)

### Step 3: Merge into `final_state_vector`
Merge Market Features + Daily Sentiment on the DatetimeIndex.

In [ ]:
final_state_vector = create_final_state_vector(market_state_df, sentiment_df)
print(f"final_state_vector Shape: {final_state_vector.shape}")
print(f"Columns: {list(final_state_vector.columns)}")
final_state_vector.head(3)

### Step 4: Chronological 80% Train / 20% Test Split

In [ ]:
split_idx = int(len(final_state_vector) * 0.8)
train_df = final_state_vector.iloc[:split_idx].copy()
test_df = final_state_vector.iloc[split_idx:].copy()

print(f"Training set (80%): {len(train_df)} candles ({len(set(train_df.index.date))} days)")
print(f"Testing set  (20%): {len(test_df)} candles ({len(set(test_df.index.date))} days)")

### Step 5: Initialize Vectorized RL Environment (File 3)
Wraps `ConstrainedTradingEnv` in `DummyVecEnv`.

In [ ]:
train_env = DummyVecEnv([lambda: ConstrainedTradingEnv(df=train_df)])
print(f"Action Space:      {train_env.action_space}")
print(f"Observation Space: {train_env.observation_space}")

### Step 6: Train PPO Agent (File 4)

In [ ]:
# Train PPO model
model = PPO(
    policy="MlpPolicy",
    env=train_env,
    learning_rate=0.0003,
    n_steps=1024,
    batch_size=64,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    verbose=1,
    seed=42,
)

# Note: In production run with total_timesteps=200000; using 5000 here for interactive demonstration
model.learn(total_timesteps=5000)
model.save(str(project_root / "ppo_trading_agent_v1"))
print("Model training complete and saved.")

### Step 7: Continuous Walk-Forward Backtesting on 20% Test Data
Executes an unbroken continuous loop across the entire test set.

In [ ]:
loaded_model = PPO.load(str(project_root / "ppo_trading_agent_v1"))
test_env = ConstrainedTradingEnv(df=test_df)
state, info = test_env.reset(seed=42)

initial_balance = test_env.initial_balance
portfolio_history = [initial_balance]
step_returns = []
timestamps = [test_df.index[0]]

done = False
step_count = 0
profitable_trades = 0

while not done:
    action, _ = loaded_model.predict(state, deterministic=True)
    state, reward, terminated, truncated, step_info = test_env.step(action)
    done = terminated or truncated
    step_count += 1

    portfolio_history.append(float(step_info["account_balance"]))
    step_returns.append(float(step_info["step_profit"]))
    timestamps.append(test_df.index[step_info["step"]])

    if step_info.get("trade_closed_profit", False):
        profitable_trades += 1

final_balance = float(step_info["account_balance"])
net_pnl = final_balance - initial_balance
net_return_pct = (net_pnl / initial_balance) * 100.0
total_fees = float(step_info["total_fees_paid"])
gross_pnl = net_pnl + total_fees
gross_return_pct = (gross_pnl / initial_balance) * 100.0
sortino = calculate_sortino_ratio(step_returns)

print(f"Backtest Finished: {step_count} candles across {step_info['days_processed']} days")
print(f"Initial Balance:  ₹{initial_balance:,.2f}")
print(f"Final Balance:    ₹{final_balance:,.2f}")
print(f"Gross PnL:        ₹{gross_pnl:+,.2f} ({gross_return_pct:+.2f}%)")
print(f"Brokerage/STT:    ₹{total_fees:,.2f}")
print(f"Net Compounded:   ₹{net_pnl:+,.2f} ({net_return_pct:+.2f}%)")
print(f"Sortino Ratio:    {sortino:.4f}")

### Step 8: Performance & Equity Curve Visualization

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(timestamps, portfolio_history, label="Compounding Portfolio Equity (₹)", color="#1f77b4", lw=2)
plt.axhline(initial_balance, color="gray", linestyle="--", alpha=0.7, label="Starting Capital (₹1,00,000)")
plt.title("Out-of-Sample Continuous Backtest Equity Curve", fontsize=13, fontweight="bold")
plt.xlabel("Intraday Timestamp")
plt.ylabel("Portfolio Value (₹)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()